# 🌊 Waterways Data Analytics

Interactive Google Colab workflow for exploratory data analysis of waterways attribute data. Upload `Waterways.csv`, run all cells, review interactive Plotly charts, and export an Excel report.

**Important:** If the CSV contains only feature attributes and no coordinates or geometry, the notebook cannot create a true geographic map or calculate waterway length/area.

In [ ]:
# ============================================================
# WATERWAYS DATA ANALYTICS | GOOGLE COLAB
# Exploratory data analysis, interactive visualisations, exports
# ============================================================

!pip -q install plotly openpyxl

import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from google.colab import files

warnings.filterwarnings("ignore")

# 1. LOAD DATA
# Upload Waterways.csv when prompted.
file_path = "/content/Waterways.csv"
if not os.path.exists(file_path):
    print("Upload Waterways.csv")
    uploaded = files.upload()
    file_path = next(iter(uploaded.keys()))

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

display(Markdown("# 🌊 Waterways Data Analytics"))
display(Markdown(
    f"**Records:** {len(df):,} &nbsp; | &nbsp; "
    f"**Columns:** {len(df.columns)}"
))
display(df.head(10))

# 2. DATA QUALITY AUDIT
display(Markdown("## 1. Data quality"))
audit = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(audit)
print("Duplicate rows:", int(df.duplicated().sum()))

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if not missing.empty:
    fig = px.bar(
        x=missing.index, y=missing.values,
        labels={"x": "Column", "y": "Missing values"},
        title="Missing values by column", text=missing.values
    )
    fig.update_layout(template="plotly_white")
    fig.show()
else:
    print("No missing cells detected.")

# 3. PREPARE DATA
data = df.copy()
for col in data.select_dtypes(include=["object", "string"]).columns:
    data[col] = data[col].astype("string").str.strip()
    data[col] = data[col].replace("", pd.NA)

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = data.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# 4. NUMERIC SUMMARY
display(Markdown("## 2. Numeric summary"))
if numeric_cols:
    display(data[numeric_cols].describe().T)

id_cols = [
    c for c in numeric_cols
    if any(k in c.lower() for k in ["id", "fid", "osm_id"])
]
analysis_numeric = [c for c in numeric_cols if c not in id_cols]

# 5. FEATURE CLASS ANALYSIS
display(Markdown("## 3. Feature class analysis"))
if "fclass" in data.columns:
    counts = (
        data["fclass"].fillna("Unknown")
        .value_counts()
        .rename_axis("Feature class")
        .reset_index(name="Count")
    )
    counts["Share (%)"] = (counts["Count"] / len(data) * 100).round(2)
    display(counts)

    fig = px.bar(
        counts.sort_values("Count"),
        x="Count", y="Feature class", orientation="h",
        color="Count", color_continuous_scale="Viridis",
        text="Count", hover_data=["Share (%)"],
        title="Waterways feature classes"
    )
    fig.update_layout(template="plotly_white", height=max(450, 30 * len(counts)))
    fig.show()

    fig = px.pie(
        counts, names="Feature class", values="Count", hole=0.42,
        title="Share of mapped feature classes"
    )
    fig.update_traces(textposition="inside", textinfo="percent+label")
    fig.update_layout(template="plotly_white")
    fig.show()

# 6. CATEGORICAL DISTRIBUTIONS
display(Markdown("## 4. Categorical distributions"))
for col in categorical_cols:
    n_unique = data[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")
    if n_unique <= 40 and col.lower() != "fclass":
        counts = (
            data[col].fillna("Unknown").value_counts().head(25)
            .rename_axis("Category").reset_index(name="Count")
        )
        fig = px.bar(
            counts.sort_values("Count"), x="Count", y="Category",
            orientation="h", text="Count",
            title=f"Category counts — {col}",
            color="Count", color_continuous_scale="Viridis"
        )
        fig.update_layout(template="plotly_white", height=500)
        fig.show()
    elif n_unique > 40:
        display(data[col].value_counts(dropna=False).head(15))

# 7. CLASS VS CODE
display(Markdown("## 5. Feature class versus code"))
if {"fclass", "code"}.issubset(data.columns):
    class_code = pd.crosstab(
        data["fclass"].fillna("Unknown"),
        data["code"].fillna(-1)
    )
    display(class_code)
    fig = px.imshow(
        class_code, aspect="auto", color_continuous_scale="Blues",
        title="Feature class vs code",
        labels={"x": "Code", "y": "Feature class", "color": "Records"}
    )
    fig.update_layout(template="plotly_white", height=600)
    fig.show()

# 8. NUMERIC DISTRIBUTIONS AND OUTLIERS
display(Markdown("## 6. Numeric distributions and outliers"))
outlier_rows = []
for col in analysis_numeric:
    fig = px.histogram(
        data, x=col, marginal="box", nbins=30,
        title=f"Distribution — {col}"
    )
    fig.update_layout(template="plotly_white")
    fig.show()

    series = pd.to_numeric(data[col], errors="coerce").dropna()
    if len(series):
        q1, q3 = series.quantile(.25), series.quantile(.75)
        iqr = q3 - q1
        n_outliers = int(((series < q1 - 1.5 * iqr) |
                          (series > q3 + 1.5 * iqr)).sum())
        outlier_rows.append({"column": col, "IQR_outliers": n_outliers})
if outlier_rows:
    display(pd.DataFrame(outlier_rows))

# 9. CORRELATION
display(Markdown("## 7. Correlation"))
if len(analysis_numeric) >= 2:
    corr = data[analysis_numeric].corr(numeric_only=True)
    fig = px.imshow(
        corr, text_auto=".2f", aspect="auto",
        color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
        title="Correlation heatmap"
    )
    fig.update_layout(template="plotly_white")
    fig.show()
else:
    print("Not enough non-ID numeric variables for meaningful correlation.")

# 10. NAMED VS UNNAMED FEATURES
display(Markdown("## 8. Named vs unnamed features"))
if "name" in data.columns:
    named = int(data["name"].notna().sum())
    unnamed = int(data["name"].isna().sum())
    name_summary = pd.DataFrame({
        "Name status": ["Named", "Unnamed"],
        "Records": [named, unnamed],
        "Percentage": [
            round(named / len(data) * 100, 2) if len(data) else 0,
            round(unnamed / len(data) * 100, 2) if len(data) else 0
        ]
    })
    display(name_summary)
    fig = px.pie(
        name_summary, names="Name status", values="Records", hole=.42,
        title="Named vs unnamed features"
    )
    fig.update_layout(template="plotly_white")
    fig.show()
    display(data["name"].value_counts().head(20).rename("Record count").to_frame())

# 11. SPATIAL READINESS
display(Markdown("## 9. Spatial data readiness"))
lower_cols = {c.lower(): c for c in data.columns}
lat_col = next((lower_cols[c] for c in
                ["latitude", "lat", "decimal_latitude"] if c in lower_cols), None)
lon_col = next((lower_cols[c] for c in
                ["longitude", "lon", "long", "decimal_longitude"] if c in lower_cols), None)

if lat_col and lon_col:
    spatial = data.copy()
    spatial[lat_col] = pd.to_numeric(spatial[lat_col], errors="coerce")
    spatial[lon_col] = pd.to_numeric(spatial[lon_col], errors="coerce")
    spatial = spatial.dropna(subset=[lat_col, lon_col])
    spatial = spatial[
        spatial[lat_col].between(-90, 90) &
        spatial[lon_col].between(-180, 180)
    ]
    if not spatial.empty:
        fig = px.scatter_map(
            spatial, lat=lat_col, lon=lon_col,
            color="fclass" if "fclass" in spatial.columns else None,
            hover_name="name" if "name" in spatial.columns else None,
            zoom=9, height=650, title="Geographic distribution of waterways"
        )
        fig.show()
else:
    print(
        "No latitude/longitude detected. The current CSV supports attribute "
        "analysis but not a true geographic map. Join verified coordinates or "
        "use a GIS file (GeoJSON, Shapefile, or GeoPackage) for spatial analysis."
    )

# 12. KEY INDICATORS AND INTERPRETATION
display(Markdown("## 10. Key indicators"))
kpis = {
    "Total records": len(data),
    "Total columns": len(data.columns),
    "Duplicate rows": int(data.duplicated().sum()),
    "Missing cells": int(data.isna().sum().sum()),
    "Missing cells (%)": round(data.isna().sum().sum() / data.size * 100, 2)
    if data.size else 0
}
if "fclass" in data.columns:
    kpis["Distinct feature classes"] = int(data["fclass"].nunique(dropna=True))
    modes = data["fclass"].mode(dropna=True)
    kpis["Most common feature class"] = modes.iloc[0] if len(modes) else "N/A"
if "name" in data.columns:
    kpis["Named features"] = int(data["name"].notna().sum())
    kpis["Unnamed features"] = int(data["name"].isna().sum())
kpi_table = pd.DataFrame(list(kpis.items()), columns=["Indicator", "Value"])
display(kpi_table)

print("Interpretation notes:")
print("- Feature counts are counts of mapped records, not waterway length or area.")
print("- OSM tags/classes reflect mapping classification, not ecological condition.")
print("- Flood-risk or environmental conclusions require spatial geometry and supporting data.")

# 13. EXPORT OUTPUTS
output_dir = "/content/waterways_analytics"
os.makedirs(output_dir, exist_ok=True)

data.to_csv(f"{output_dir}/waterways_cleaned.csv", index=False)
audit.to_csv(f"{output_dir}/data_quality_audit.csv")
kpi_table.to_csv(f"{output_dir}/waterways_kpis.csv", index=False)

if "fclass" in data.columns:
    (
        data["fclass"].fillna("Unknown").value_counts()
        .rename_axis("Feature class").reset_index(name="Count")
        .to_csv(f"{output_dir}/feature_class_summary.csv", index=False)
    )

with pd.ExcelWriter(
    f"{output_dir}/waterways_analytics_report.xlsx", engine="openpyxl"
) as writer:
    data.to_excel(writer, sheet_name="Cleaned Data", index=False)
    audit.to_excel(writer, sheet_name="Data Quality")
    kpi_table.to_excel(writer, sheet_name="KPIs", index=False)
    if "fclass" in data.columns:
        (
            data["fclass"].fillna("Unknown").value_counts()
            .rename_axis("Feature class").reset_index(name="Count")
            .to_excel(writer, sheet_name="Feature Summary", index=False)
        )

print("Analysis complete. Files saved to:", output_dir)
files.download(f"{output_dir}/waterways_analytics_report.xlsx")
